Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

Load & Understand Data

In [22]:
df = pd.read_csv("Twitter_Data.csv")  # change file name

print(df.head())
print(df.info())

# Class distribution
print(df['category'].value_counts())

                                          clean_text  category
0  when modi promised “minimum government maximum...      -1.0
1  talk all the nonsense and continue all the dra...       0.0
2  what did just say vote for modi  welcome bjp t...       1.0
3  asking his supporters prefix chowkidar their n...       1.0
4  answer who among these the most powerful world...       1.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB
None
category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64


Check for Nan value in dataset

In [23]:
print(df['category'].isna().sum())

7


Deleting Nan values row.

In [24]:
df = df.dropna(subset=['category'])

Again Checking

In [25]:
print(df['category'].isna().sum())

0


NLP Preprocessing

In [26]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    tokens = text.split()

    # Remove stopwords & Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

Apply preprocessing:

In [27]:
df['clean_text'] = df['clean_text'].fillna("").astype(str)
df['re_clean_text'] = df['clean_text'].apply(preprocess_text)

print(df[['clean_text', 're_clean_text']].head())

                                          clean_text  \
0  when modi promised “minimum government maximum...   
1  talk all the nonsense and continue all the dra...   
2  what did just say vote for modi  welcome bjp t...   
3  asking his supporters prefix chowkidar their n...   
4  answer who among these the most powerful world...   

                                       re_clean_text  
0  modi promised “minimum government maximum gove...  
1             talk nonsense continue drama vote modi  
2  say vote modi welcome bjp told rahul main camp...  
3  asking supporter prefix chowkidar name modi gr...  
4  answer among powerful world leader today trump...  


Bag of Words (BoW)

In [28]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['re_clean_text'])

TF-IDF

In [29]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['re_clean_text'])

Train-Test Split

In [30]:
y = df['category']

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

Model Evaluation

In [31]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

Logistic Regression

In [32]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)

print("Logistic Regression (TF-IDF):")
evaluate_model(lr, X_test_tfidf, y_test)

Logistic Regression (TF-IDF):
Accuracy: 0.8798895536125173
Precision: 0.881566203098852
Recall: 0.8798895536125173
F1 Score: 0.8786791943449359

Classification Report:
               precision    recall  f1-score   support

        -1.0       0.87      0.76      0.81      7230
         0.0       0.85      0.96      0.90     10961
         1.0       0.91      0.88      0.90     14404

    accuracy                           0.88     32595
   macro avg       0.88      0.87      0.87     32595
weighted avg       0.88      0.88      0.88     32595



Naive Bayes

In [33]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

print("Naive Bayes (BoW):")
evaluate_model(nb, X_test_bow, y_test)

Naive Bayes (BoW):
Accuracy: 0.7715907347752723
Precision: 0.7734772209413853
Recall: 0.7715907347752723
F1 Score: 0.7717493592999026

Classification Report:
               precision    recall  f1-score   support

        -1.0       0.69      0.71      0.70      7230
         0.0       0.82      0.75      0.79     10961
         1.0       0.78      0.82      0.80     14404

    accuracy                           0.77     32595
   macro avg       0.76      0.76      0.76     32595
weighted avg       0.77      0.77      0.77     32595



Decision Tree

In [34]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

print("Decision Tree (TF-IDF):")
evaluate_model(dt, X_test_tfidf, y_test)

Decision Tree (TF-IDF):
Accuracy: 0.7997852431354502
Precision: 0.798492084912589
Recall: 0.7997852431354502
F1 Score: 0.7988934020409116

Classification Report:
               precision    recall  f1-score   support

        -1.0       0.69      0.66      0.68      7230
         0.0       0.83      0.87      0.85     10961
         1.0       0.83      0.82      0.82     14404

    accuracy                           0.80     32595
   macro avg       0.78      0.78      0.78     32595
weighted avg       0.80      0.80      0.80     32595



Comparison Table

In [35]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Vectorization': ['TF-IDF', 'BoW', 'TF-IDF'],
    'Accuracy': [0.88, 0.77, 0.80]  # replace with actual results
})

print(results)

                 Model Vectorization  Accuracy
0  Logistic Regression        TF-IDF      0.88
1          Naive Bayes           BoW      0.77
2        Decision Tree        TF-IDF      0.80


# **Comparison & Insights**

Best Model- Logistic Regression

Best Vectorization - TF-IDF

Preprocessing - Removing Nan value, Removing stopwords improved clarity, Lemmatization helped normalize words.

Observations - Naive Bayes is fast but assumes independence, Decision Tree overfits on text data.